In [ ]:
from PipelineTS import Agent
import pandas as pd

# agent配置，先做deepseek的适配
# kwargs为provider的python api关键词参数
agent = Agent(api_key="sk-xxx", provider="deepseek", model="deepseek-v4-pro", 
              base_url="https://api.deepseek.com", api_type="openai", stream=False,
              reasoning_effort="high", **kwargs)

file_path = "./test.csv"

# 支持多个时间列、多个目标列
# 支持多个历史协变量
# 支持同时预测多个时间序列，获取序列之间的关系增加预测的准确性
agent.init(fps=[file_path], id_col=None, time_cols=['time_col'], targets=['target_col'], past_covariates=['col_1', 'col_2'])

agent.analysis(result_path="./result_path.md")

# llm_fit_only=True，仅使用provider llm进行预测，pipelinets提供除模型外的所有工具（包括预处理、特征增强等），
#   需要综合业界sota时间序列llm预测的策略和方案，选择适合的或者进行优化改造。
# llm_fit_only=False，使用provider llm和pipelinets所提供的模型、工具进行预测，择优选择最佳方案。
agent.fit(refit=True, lag_windows="auto", train_rate=0.7, 
          epochs="auto", lr_scheduler="auto")

# strategy.pts 可以是模型，也可以是大语言模型得出的数据规律等可以复现或者接近复现预测结果的数据，并且可以用来预测接下来任意n个时间点。
# 并且可以复用到不同provider llm中
agent.save("./strategy.pts")

future_covariates = pd.read_csv("./future_covariates.csv")

# future_covariates 为未来可确定的协变量，不一定等同于past_covariates列
# reasonable输出预测依据
agent.predict(n=30, future_covariates=future_covariates, reasonable=True, reason_result_path="./reason_result.md")

In [2]:
from PipelineTS import Agent
from PipelineTS.dataset import BuiltInSeriesData
import pandas as pd
from pathlib import Path
import os

source = BuiltInSeriesData(print_file_list=False)
wrapper = source["AirPassengers"]

df = pd.DataFrame(wrapper).copy()
time_col = wrapper.time_col
target_col = wrapper.target_col

df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
df = df.dropna(subset=[time_col, target_col]).sort_values(time_col).reset_index(drop=True)

horizon = 12
train_df = df.iloc[:-horizon].copy()

work_dir = Path("./agent_builtin_case")
work_dir.mkdir(parents=True, exist_ok=True)

train_path = work_dir / "air_passengers_train.csv"
train_df.to_csv(train_path, index=False)

agent = Agent(
    api_key=os.environ.get("PIPELINETS_AGENT_API_KEY", "sk-b9e3f87db2ca466a805d0bcd03b33ebc"),
    provider="deepseek",
    model="deepseek-v4-pro",
    base_url="https://api.deepseek.com",
    api_type="openai",
    stream=False,
)

agent.init(
    fps=[train_path],
    id_col=None,
    time_cols=[time_col],
    targets=[target_col],
)

agent.fit(
    refit=True,
    lag_windows="auto",
    train_rate=0.7,

    # 关键：让候选里一定出现 PipelineTS
    agentic_force_pipeline_eval=True,
    include_models="light",
    time_limit=10,
)

task_key = next(iter(agent.task_states))
state = agent.task_states[task_key]

print("selected:", state["selected"])
print("local_metric:", state["llm_metric"])
print("pipeline_metric:", state["pipeline_metric"])
print("llm_decision:", state["agentic_plan"]["llm_decision"])
print("candidates:", state["agentic_plan"]["candidates"])
print("tool_trace:")
for step in state["agentic_plan"]["tool_trace"]:
    print("-", step["tool"], step["result"])

selected: local
local_metric: 25.092436974789923
pipeline_metric: inf
llm_decision: {'error': "BadRequestError: Error code: 400 - {'error': {'message': 'The `reasoning_content` in the thinking mode must be passed back to the API.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}"}
candidates: {'local': {'type': 'deterministic_local_ensemble', 'validation_mae': 25.092437}, 'pipelinets': {'type': 'pipelinets_smartrouter', 'validation_mae': None, 'config': {'preset': 'fast', 'time_limit': 30}}}
tool_trace:
- inspect_task {'task_key': 'Month::Passengers', 'rows': 132, 'train_rows': 120, 'valid_rows': 12, 'horizon': 12, 'lag_window': 12, 'id_col': None, 'known_covariates': [], 'analysis_context': {}, 'local_diagnostics': {'detected_period': 12, 'calendar_period': 12, 'trend_strength': 0.026261}, 'available_tools': ['inspect_task', 'fit_local_strategy', 'fit_pipelinets_strategy', 'compare_strategies', 'select_strategy']}
- fit_local_strategy {'candidate': 'l